# NFL 4th Down — Coach Grading
Load the graded play-by-play, merge head coach names from nflverse schedules,
compute DQS and ODR per coach, and produce ranked charts.

**Inputs:** `data/fourth_downs_graded.parquet`  
**Outputs:** `outputs/coach_grades.csv`, three charts in `outputs/figures/`

| Metric | Definition | Direction |
|--------|------------|-----------|
| **DQS** | Mean decision gap (optimal_wpa − actual_wpa) per play | Lower = better |
| **ODR** | % of plays where coach made the historically optimal call | Higher = better |
| **Go rate** | % of 4th down plays where coach chose to go for it | Context-dependent |

In [ ]:
import sys
sys.path.append('../src')

import os
import io
import requests
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

from grading import grade_coaches

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

DATA_DIR  = '../data/'
SAVE_DIR  = '../outputs/figures/'
OUT_DIR   = '../outputs/'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)

CAMPBELL        = 'Dan Campbell'
CAMPBELL_COLOR  = '#C5203B'   # punchy red for highlighting
DEFAULT_GOOD    = '#2979FF'   # blue — best coaches
DEFAULT_BAD     = '#EF5350'   # red  — worst coaches
DEFAULT_SCATTER = '#90A4AE'   # muted grey — background coaches

## 1. Load Graded Play-by-Play

In [ ]:
fourth = pd.read_parquet(DATA_DIR + 'fourth_downs_graded.parquet')

print(f'Shape: {fourth.shape}')
print(f'Seasons: {fourth["season"].min()} – {fourth["season"].max()}')
print(f'Decisions: {fourth["decision"].value_counts().to_dict()}')
print(f'Plays with decision_gap: {fourth["decision_gap"].notna().sum():,} ({fourth["decision_gap"].notna().mean()*100:.1f}%)')
fourth[['play_id','game_id','season','posteam','home_team','away_team','decision','decision_gap']].head(3)

## 2. Load Coach Names from nflverse
nflverse schedules (`games.csv`) carries `home_coach` and `away_coach` for every game 1999–2025.
We join on `game_id`, then select the right coach based on whether the possessing team was home or away.

In [ ]:
GAMES_URL   = 'https://github.com/nflverse/nflverse-data/releases/download/schedules/games.csv'
COACH_CACHE = DATA_DIR + 'raw/games_coaches.parquet'

if os.path.exists(COACH_CACHE):
    games = pd.read_parquet(COACH_CACHE)
    print(f'Loaded from cache: {len(games):,} games')
else:
    print('Downloading nflverse schedule data...')
    resp = requests.get(GAMES_URL, headers={'User-Agent': 'nfl-4thdown/1.0'}, timeout=60)
    resp.raise_for_status()
    games = pd.read_csv(io.BytesIO(resp.content))
    games.to_parquet(COACH_CACHE, index=False)
    print(f'Downloaded and cached {len(games):,} games')

print(f'Seasons: {games["season"].min()} – {games["season"].max()}')
print(f'NaN coaches: home={games["home_coach"].isna().sum()}, away={games["away_coach"].isna().sum()}')
games[['season','week','game_id','home_team','home_coach','away_team','away_coach']].head(3)

In [ ]:
# Merge only the coach columns — home_team/away_team are already in `fourth`
coach_lookup = games[['game_id', 'home_coach', 'away_coach']].drop_duplicates('game_id')

fourth = fourth.merge(coach_lookup, on='game_id', how='left')

# Assign coaching credit to the possessing team
fourth['coach_name'] = np.where(
    fourth['posteam'] == fourth['home_team'],
    fourth['home_coach'],
    fourth['away_coach']
)
fourth = fourth.drop(columns=['home_coach', 'away_coach'])

matched = fourth['coach_name'].notna().sum()
print(f'Plays with coach name: {matched:,} / {len(fourth):,} ({matched/len(fourth)*100:.1f}%)')
print(f'Unique coaches: {fourth["coach_name"].nunique()}')

# Confirm Campbell is present
dc_plays = fourth[fourth['coach_name'] == CAMPBELL]
print(f'\nDan Campbell plays: {len(dc_plays):,} (seasons {dc_plays["season"].min()}–{dc_plays["season"].max()})')

## 3. Compute Coach Grades
Minimum 50 graded decisions to qualify. Uses `grade_coaches()` from `src/grading.py`.

In [ ]:
grades = grade_coaches(fourth, min_decisions=50, coach_col='coach_name')

print(f'Coaches qualifying (≥ 50 graded decisions): {len(grades)}')
print(f'DQS range: {grades["dqs"].min():.4f} – {grades["dqs"].max():.4f}')
print(f'ODR range: {grades["odr"].min()*100:.1f}% – {grades["odr"].max()*100:.1f}%')
print(f'Go rate range: {grades["go_rate"].min()*100:.1f}% – {grades["go_rate"].max()*100:.1f}%')
grades.head(3)

In [ ]:
# ── Top 10 best coaches (lowest DQS) ─────────────────────────────────────────
display_cols = ['coach_name','total_decisions','dqs','odr','go_rate','seasons','dqs_rank']

print('Top 10 — Best Decision Quality (lowest mean decision gap):')
print(
    grades.head(10)[display_cols]
    .assign(
        dqs      = lambda x: x['dqs'].map('{:.4f}'.format),
        odr      = lambda x: (x['odr'] * 100).map('{:.1f}%'.format),
        go_rate  = lambda x: (x['go_rate'] * 100).map('{:.1f}%'.format),
    )
    .to_string(index=False)
)

In [ ]:
# ── Bottom 10 worst coaches (highest DQS) ────────────────────────────────────
print('Bottom 10 — Worst Decision Quality (highest mean decision gap):')
print(
    grades.tail(10)[display_cols]
    .assign(
        dqs      = lambda x: x['dqs'].map('{:.4f}'.format),
        odr      = lambda x: (x['odr'] * 100).map('{:.1f}%'.format),
        go_rate  = lambda x: (x['go_rate'] * 100).map('{:.1f}%'.format),
    )
    .to_string(index=False)
)

In [ ]:
# ── Dan Campbell's exact position ─────────────────────────────────────────────
dc = grades[grades['coach_name'] == CAMPBELL]
if dc.empty:
    print('Dan Campbell not found — check min_decisions threshold or coach name spelling')
else:
    n = len(grades)
    row = dc.iloc[0]
    pct = row['dqs_rank'] / n * 100
    print(f'Dan Campbell')
    print(f'  DQS rank  : {int(row["dqs_rank"])} / {n}  ({pct:.0f}th percentile — lower rank = better)')
    print(f'  ODR rank  : {int(row["odr_rank"])} / {n}  (lower rank = better)')
    print(f'  DQS (mean gap): {row["dqs"]:.4f}')
    print(f'  ODR           : {row["odr"]*100:.1f}%')
    print(f'  Go-for-it rate: {row["go_rate"]*100:.1f}%')
    print(f'  Total decisions: {int(row["total_decisions"]):,}')
    print(f'  Seasons active : {row["seasons"]}')

## 4. Chart — Top 15 & Bottom 15 Coaches by DQS
Horizontal bar charts. Lower DQS (mean decision gap) = closer to historically optimal decisions.

In [ ]:
def bar_colors(names, default, highlight_name=CAMPBELL, highlight_color=CAMPBELL_COLOR):
    """Return a color list, substituting highlight_color for highlight_name."""
    return [highlight_color if n == highlight_name else default for n in names]


top15 = grades.head(15).sort_values('dqs', ascending=True)   # low DQS = good, plot left-to-right
bot15 = grades.tail(15).sort_values('dqs', ascending=False)  # high DQS = bad, longest bar at top

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
fig.suptitle(
    '4th Down Decision Quality — Head Coach Rankings (1999–2025)',
    fontsize=13, fontweight='bold', y=1.01
)

# ── Left: Best 15 ──────────────────────────────────────────────────────────
ax = axes[0]
colors = bar_colors(top15['coach_name'], DEFAULT_GOOD)
bars = ax.barh(top15['coach_name'], top15['dqs'] * 100, color=colors, height=0.65)
ax.set_xlabel('Mean Decision Gap (WPA pts)', fontsize=10)
ax.set_title('Best 15  —  Lowest Decision Gap', fontsize=11, fontweight='bold', color=DEFAULT_GOOD)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
for bar, val in zip(bars, top15['dqs'] * 100):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha='left', fontsize=7.5)

# ── Right: Worst 15 ────────────────────────────────────────────────────────
ax2 = axes[1]
colors2 = bar_colors(bot15['coach_name'], DEFAULT_BAD)
bars2 = ax2.barh(bot15['coach_name'], bot15['dqs'] * 100, color=colors2, height=0.65)
ax2.set_xlabel('Mean Decision Gap (WPA pts)', fontsize=10)
ax2.set_title('Worst 15  —  Highest Decision Gap', fontsize=11, fontweight='bold', color=DEFAULT_BAD)
ax2.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
for bar, val in zip(bars2, bot15['dqs'] * 100):
    ax2.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', ha='left', fontsize=7.5)

# Campbell legend (only if he appears in either panel)
campbell_visible = (
    CAMPBELL in top15['coach_name'].values or
    CAMPBELL in bot15['coach_name'].values
)
if campbell_visible:
    fig.legend(
        handles=[mpatches.Patch(color=CAMPBELL_COLOR, label='Dan Campbell')],
        loc='lower center', ncol=1, fontsize=10, bbox_to_anchor=(0.5, -0.03)
    )

plt.tight_layout()
plt.savefig(SAVE_DIR + '06_coach_dqs_rankings.png', bbox_inches='tight')
plt.show()
print('Saved: 06_coach_dqs_rankings.png')

## 5. Chart — Go-For-It Rate vs DQS
Does aggression correlate with decision quality?  
Modern analytics argues going for it more often improves outcomes — this tests that hypothesis.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# All other coaches
others = grades[grades['coach_name'] != CAMPBELL]
ax.scatter(
    others['go_rate'] * 100,
    others['dqs'] * 100,
    color=DEFAULT_SCATTER, alpha=0.65, s=40, zorder=2, label='Other coaches'
)

# Trend line
z = np.polyfit(others['go_rate'] * 100, others['dqs'] * 100, 1)
p = np.poly1d(z)
x_range = np.linspace(others['go_rate'].min() * 100, others['go_rate'].max() * 100, 100)
ax.plot(x_range, p(x_range), color='#546E7A', linewidth=1.2, linestyle='--',
        alpha=0.7, label=f'Trend (slope={z[0]:.3f})')

# Campbell
dc = grades[grades['coach_name'] == CAMPBELL]
if not dc.empty:
    ax.scatter(
        dc['go_rate'] * 100, dc['dqs'] * 100,
        color=CAMPBELL_COLOR, s=140, zorder=5, marker='*', label='Dan Campbell'
    )
    ax.annotate(
        'Dan Campbell',
        xy=(dc['go_rate'].values[0] * 100, dc['dqs'].values[0] * 100),
        xytext=(8, 6), textcoords='offset points',
        fontsize=9, color=CAMPBELL_COLOR, fontweight='bold'
    )

ax.set_xlabel('Go-For-It Rate (%)', fontsize=11)
ax.set_ylabel('Mean Decision Gap — DQS (WPA pts)', fontsize=11)
ax.set_title(
    'Does Aggression Improve Decision Quality?\nGo-For-It Rate vs Mean Decision Gap per Coach',
    fontsize=12, fontweight='bold'
)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(fontsize=9)

# Directional annotation
ax.text(0.01, 0.99, '← Better decisions (lower gap)',
        transform=ax.transAxes, va='top', ha='left', fontsize=8, color='#546E7A')

plt.tight_layout()
plt.savefig(SAVE_DIR + '07_aggression_vs_quality.png', bbox_inches='tight')
plt.show()
print('Saved: 07_aggression_vs_quality.png')

# Print correlation
corr = grades['go_rate'].corr(grades['dqs'])
print(f'\nCorrelation (go_rate vs DQS): {corr:.3f}')
print('Negative = more aggressive coaches have lower (better) DQS')

## 6. Chart — ODR vs Total Decisions (Volume vs Accuracy)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

others = grades[grades['coach_name'] != CAMPBELL]

# Bubble size proportional to go-for-it rate (so we can encode 3 variables)
bubble_scale = (others['go_rate'] * 100).clip(lower=2)
ax.scatter(
    others['total_decisions'],
    others['odr'] * 100,
    s=bubble_scale ** 1.5,
    color=DEFAULT_SCATTER, alpha=0.55, zorder=2, label='Other coaches (size = go rate)'
)

# League average ODR line
league_odr = grades['odr'].mean() * 100
ax.axhline(league_odr, color='#546E7A', linewidth=1, linestyle='--', alpha=0.7,
           label=f'League avg ODR ({league_odr:.1f}%)')

# Campbell
dc = grades[grades['coach_name'] == CAMPBELL]
if not dc.empty:
    dc_go = dc['go_rate'].values[0] * 100
    ax.scatter(
        dc['total_decisions'], dc['odr'] * 100,
        s=max(dc_go ** 1.5, 80), color=CAMPBELL_COLOR, zorder=5,
        marker='*', label='Dan Campbell'
    )
    ax.annotate(
        'Dan Campbell',
        xy=(dc['total_decisions'].values[0], dc['odr'].values[0] * 100),
        xytext=(8, -12), textcoords='offset points',
        fontsize=9, color=CAMPBELL_COLOR, fontweight='bold'
    )

ax.set_xlabel('Total Graded Decisions', fontsize=11)
ax.set_ylabel('Optimal Decision Rate — ODR (%)', fontsize=11)
ax.set_title(
    'Volume vs Accuracy: Do Coaches Improve Over More Decisions?\n'
    'Bubble size = go-for-it rate',
    fontsize=12, fontweight='bold'
)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(SAVE_DIR + '08_odr_vs_volume.png', bbox_inches='tight')
plt.show()
print('Saved: 08_odr_vs_volume.png')

# Print correlation
corr = grades['total_decisions'].corr(grades['odr'])
print(f'\nCorrelation (total_decisions vs ODR): {corr:.3f}')

## 7. Summary Table — All Qualifying Coaches
Quick look at the full leaderboard before saving.

In [ ]:
summary = grades[display_cols].assign(
    dqs     = lambda x: x['dqs'].map('{:.4f}'.format),
    odr     = lambda x: (x['odr'] * 100).map('{:.1f}%'.format),
    go_rate = lambda x: (x['go_rate'] * 100).map('{:.1f}%'.format),
)

print(f'Full leaderboard ({len(grades)} coaches):')
print(summary.to_string(index=False))

## 8. Save Coach Grades

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
out_path = OUT_DIR + 'coach_grades.csv'
grades.to_csv(out_path, index=False)

print(f'Saved {len(grades):,} coach records → {out_path}')
print(f'Columns: {grades.columns.tolist()}')

# Sanity check — confirm Campbell is in the file
dc_check = grades[grades['coach_name'] == CAMPBELL]
if not dc_check.empty:
    print(f'\nDan Campbell confirmed in output (rank {int(dc_check["dqs_rank"].values[0])} of {len(grades)})')